# Datawizards · Preparación y calidad de datos (Guía 4)

**Qué hace este notebook:** toma los datos abiertos (RUNT, DANE y OpenStreetMap), los limpia, mide su calidad y calcula los indicadores por municipio para el índice de priorización de carga pública de vehículos eléctricos. Cada paso deja archivos en carpetas que se descargan al final.

| Paso | Qué hace | Salida principal |
|---|---|---|
| 1–2 | Instala librerías y crea los scripts | `*.py` |
| 3 | Localiza los archivos subidos | variables `EV`, `PARQUE`, `DANE` |
| 4 | Limpia RUNT y cruza con DIVIPOLA (DANE) | `prep/runt_anual.csv`, `prep/parque_liviano.csv` |
| 5 | Calidad, limpieza y EDA de demanda | `prep/eda/` (indicadores, correlaciones, VIF, clusters, `resumen.md`) |
| 6 | Descarga y explora OpenStreetMap | `osm/capas_gpkg.csv`, `osm/fclass_*.csv` |
| 7 | Arma límites municipales desde OSM (plan B del MGN) | `osm/mgn_osm.gpkg` |
| 8 | Accesibilidad por municipio (vías, comercio, parqueaderos, combustible) | `osm/accesibilidad_municipal.csv` |
| 9 | EDA final con demanda + accesibilidad | `prep/eda_acc/` |
| 10 | Descarga el zip de resultados | `resultados_datawizards.zip` |

**Antes de empezar**, arrastre al panel de archivos de Colab (ícono de carpeta) estos tres archivos, con sus nombres originales:
1. CSV de *Vehículos eléctricos e híbridos matriculados* (RUNT 2.0, datos.gov.co `ntxy-q9ix`).
2. CSV de *Crecimiento del parque automotor* (RUNT 2.0, `u3vn-bdcy`).
3. `PPED-AreaMun-2018-2042_VP.xlsx` (proyecciones de población DANE).

Ejecute las celdas **en orden**. Las celdas con `%%writefile` solo crean archivos y no calculan nada.

> **Supuestos que el equipo debe validar** (están marcados con `# AJUSTAR`): combustible `ELECTRICO`, clases `AUTOMOVIL, CAMIONETA, CAMPERO`, año de población 2026 y último año completo 2025.

## 1. Instalar librerías

In [1]:
# Librerías que Colab no trae: coincidencia aproximada de nombres, tablas y datos geográficos
!pip install rapidfuzz tabulate openpyxl geopandas pyogrio -q
import glob, os
print("Archivos en la carpeta actual:", os.listdir("."))

Archivos en la carpeta actual: ['.config', 'sample_data']


## 2. Crear los scripts y las tablas de alias
Cada celda escribe un archivo en el entorno de Colab. Si cambia un script, vuelva a ejecutar su celda antes de usarlo.

**2a. `prep_runt_datosgov.py`** limpia el RUNT y la población del DANE y los cruza por código DIVIPOLA.

In [2]:
%%writefile prep_runt_datosgov.py
#!/usr/bin/env python3
"""Prepara los datos del RUNT (datos.gov.co) y la población del DANE para el análisis Datawizards.

Archivos reales usados:
  ev.csv     VEHÍCULOS ELÉCTRICOS E HÍBRIDOS MATRICULADOS EN COLOMBIA - RUNT 2.0 (id ntxy-q9ix)
             COMBUSTIBLE, MODELO, AÑO_REGISTRO, CLASE, CARROCERIA, MUNICIPIO, DEPARTAMENTO, CANTIDAD
  parque.csv CRECIMIENTO DEL PARQUE AUTOMOTOR RUNT2.0 (id u3vn-bdcy)
             NOMBRE_DEPARTAMENTO, NOMBRE_MUNICIPIO, NOMBRE_SERVICIO, ESTADO_DEL_VEHICULO,
             NOMBRE_DE_LA_CLASE, FECHA DE REGISTRO (año), CANTIDAD, MES/AÑO DE PUBLICACIÓN
  dane.xlsx  PPED-AreaMun-2018-2042 (hoja PobMunicipalxÁrea: DP, DPNOM, MPIO, DPMP, AÑO, ÁREA GEOGRÁFICA, TOTAL)

Pasos:
  1) python prep_runt_datosgov.py dane     --xlsx dane.xlsx --anio 2026 --out salida_prep
  2) python prep_runt_datosgov.py explorar --runt ev.csv --out salida_prep
  3) python prep_runt_datosgov.py preparar --runt ev.csv --dane salida_prep/dane_municipios.csv \
         --combustibles ELECTRICO --clases "AUTOMOVIL,CAMIONETA,CAMPERO,BUS,BUSETA,MICROBUS" --out salida_prep
  4) python prep_runt_datosgov.py parque   --runt parque.csv --dane salida_prep/dane_municipios.csv \
         --clases "AUTOMOVIL,CAMIONETA,CAMPERO" --out salida_prep

Alerta de formato: en ev.csv algunas CANTIDAD vienen con separador de miles ("1,164"). Convertirlas
con coerción las vuelve vacías y se pierden las filas más grandes (casi la mitad de los vehículos).
Este script las lee correctamente y reporta cuántas encontró.
"""
import argparse
import os
import re
import unicodedata

import pandas as pd
from rapidfuzz import fuzz, process

UMBRAL_MUNICIPIO = 92  # similitud mínima dentro del mismo departamento


def norm(t):
    """Minúsculas, sin tildes ni puntuación, sin 'distrito', 'd.c.' o 'municipio de'."""
    t = unicodedata.normalize("NFKD", str(t)).encode("ascii", "ignore").decode().lower()
    t = re.sub(r"[^a-z0-9 ]", " ", t)
    t = re.sub(r"\b(distrito|capital|dc|d c|municipio de)\b", " ", t)
    return re.sub(r"\s+", " ", t).strip()


def a_numero(serie):
    """Convierte texto a número aceptando separador de miles ('1,164'). Devuelve (serie, n_con_separador)."""
    t = serie.astype(str).str.strip()
    n_sep = int(t.str.contains(",", regex=False).sum())
    return pd.to_numeric(t.str.replace(",", "", regex=False), errors="coerce"), n_sep


RENOMBRAR = {"ano_registro": "anio", "a_o_registro": "anio", "fecha_de_registro": "anio",
             "nombre_departamento": "departamento", "nombre_municipio": "municipio",
             "nombre_de_la_clase": "clase", "nombre_servicio": "servicio", "estado_del_vehiculo": "estado"}


def leer(path):
    df = pd.read_csv(path, dtype=str)
    df.columns = [norm(c).replace(" ", "_") for c in df.columns]
    return df.rename(columns=RENOMBRAR)


def dane_a_csv(a):
    os.makedirs(a.out, exist_ok=True)
    raw = pd.read_excel(a.xlsx, sheet_name=a.hoja, header=None, dtype=str)
    fila = raw.index[raw.apply(lambda f: f.astype(str).str.strip().eq("MPIO").any(), axis=1)][0]
    d = pd.read_excel(a.xlsx, sheet_name=a.hoja, header=fila, dtype=str).dropna(subset=["MPIO"])
    d = d[(d["ÁREA GEOGRÁFICA"].str.strip() == "Total") & (pd.to_numeric(d["AÑO"], errors="coerce") == a.anio)]
    out = pd.DataFrame({"divipola": d["MPIO"].str.zfill(5), "departamento": d["DPNOM"], "municipio": d["DPMP"],
                        "poblacion": pd.to_numeric(d["TOTAL"], errors="coerce")})
    out.to_csv(os.path.join(a.out, "dane_municipios.csv"), index=False)
    print(f"dane_municipios.csv: {len(out)} municipios, población {a.anio}: {out['poblacion'].sum():,.0f}")


def explorar(a):
    os.makedirs(a.out, exist_ok=True)
    r = leer(a.runt)
    r["cantidad"], n_sep = a_numero(r["cantidad"])
    print(f"Filas: {len(r)} | CANTIDAD con separador de miles: {n_sep} | CANTIDAD no numérica: {int(r['cantidad'].isna().sum())}")
    print("Vacíos por columna:\n", r.isna().sum())
    for col, f in (("combustible", "valores_combustible.csv"), ("clase", "valores_clase.csv"), ("anio", "anios.csv")):
        (r.groupby(col, dropna=False)["cantidad"].agg(["count", "sum"]).sort_values("sum", ascending=False)
         .to_csv(os.path.join(a.out, f)))
    print(r.groupby("combustible")["cantidad"].sum())
    print("Revise los CSV de valores antes de preparar.")


def cruzar(r, dane, alias=None):
    d = dane.copy()
    d["k_dep"], d["k_mun"] = d["departamento"].map(norm), d["municipio"].map(norm)
    d["divipola"] = d["divipola"].astype(str).str.zfill(5)
    llave = {(a, b): c for a, b, c in zip(d["k_dep"], d["k_mun"], d["divipola"])}
    por_dep = {k: g for k, g in d.groupby("k_dep")}
    deps = list(por_dep)
    salida = {}
    for dep_raw, mun_raw in r[["departamento", "municipio"]].drop_duplicates().itertuples(index=False):
        kd, km = norm(dep_raw), norm(mun_raw)
        if alias is not None and (kd, km) in alias:
            salida[(dep_raw, mun_raw)] = (alias[(kd, km)], "alias", 100)
            continue
        if (kd, km) in llave:
            salida[(dep_raw, mun_raw)] = (llave[(kd, km)], "exacto", 100)
            continue
        dep_match = kd if kd in por_dep else (process.extractOne(kd, deps, scorer=fuzz.token_set_ratio) or (None, 0))[0]
        if dep_match in por_dep:
            g = por_dep[dep_match]
            m = process.extractOne(km, list(g["k_mun"]), scorer=fuzz.ratio)
            if m and m[1] >= UMBRAL_MUNICIPIO:
                salida[(dep_raw, mun_raw)] = (g.iloc[m[2]]["divipola"], "difuso", round(m[1], 1))
                continue
        salida[(dep_raw, mun_raw)] = (None, "sin_cruce", 0)
    return salida


def preparar(a):
    os.makedirs(a.out, exist_ok=True)
    r = leer(a.runt)
    dane = pd.read_csv(a.dane, dtype={"divipola": str})
    r["cantidad"], n_sep = a_numero(r["cantidad"])
    print(f"CANTIDAD con separador de miles corregidas: {n_sep}")
    r["anio"] = pd.to_numeric(r["anio"], errors="coerce")
    # calidad del archivo crudo (antes de filtrar): vacíos y rangos
    crudo = [(c, int(r[c].isna().sum()), f"{r[c].isna().mean():.2%}") for c in r.columns]
    pd.DataFrame(crudo, columns=["columna", "vacios", "pct"]).to_csv(os.path.join(a.out, "calidad_runt_crudo.csv"), index=False)
    print("Vacíos en el archivo crudo:", {c: v for c, v, _ in crudo if v})
    comb = [norm(x) for x in a.combustibles.split(",")]
    clases = [norm(x) for x in a.clases.split(",")]
    n0 = len(r)
    r = r[r["combustible"].map(norm).isin(comb) & r["clase"].map(norm).isin(clases)].copy()
    print(f"Filas del conjunto: {n0}; tras filtrar combustible y clase: {len(r)}")
    alias = None
    if a.alias:  # CSV: departamento, municipio (como vienen en el RUNT), divipola
        al = pd.read_csv(a.alias, dtype=str)
        alias = {(norm(x), norm(y)): z.zfill(5) for x, y, z in zip(al["departamento"], al["municipio"], al["divipola"])}
    mapa = cruzar(r, dane, alias)
    r["divipola"] = [mapa[(d, m)][0] for d, m in zip(r["departamento"], r["municipio"])]
    rep = pd.DataFrame([(d, m, *v) for (d, m), v in mapa.items()],
                       columns=["departamento", "municipio", "divipola", "tipo_cruce", "similitud"])
    rep = rep.merge(r.groupby(["departamento", "municipio"])["cantidad"].sum().rename("vehiculos").reset_index(), how="left")
    rep.sort_values(["tipo_cruce", "vehiculos"], ascending=[True, False]).to_csv(os.path.join(a.out, "reporte_cruce_municipios.csv"), index=False)
    sin = r["divipola"].isna()
    print(f"Filas sin cruce con DANE: {int(sin.sum())} ({r.loc[sin, 'cantidad'].sum():,.0f} vehículos). Revise reporte_cruce_municipios.csv")
    print("Cruces difusos para validar a mano:", int((rep["tipo_cruce"] == "difuso").sum()))
    print("Filas filtradas con cantidad vacía (no suman):", int(r["cantidad"].isna().sum()))
    out = (r.dropna(subset=["divipola"]).groupby(["divipola", "anio"], as_index=False)["cantidad"].sum())
    out.to_csv(os.path.join(a.out, "runt_anual.csv"), index=False)
    print("runt_anual.csv:", len(out), "filas")


def parque(a):
    """Parque activo total (conjunto 'Crecimiento del parque automotor RUNT 2.0') por municipio, solo clases livianas."""
    os.makedirs(a.out, exist_ok=True)
    r = leer(a.runt)
    dane = pd.read_csv(a.dane, dtype={"divipola": str})
    r["cantidad"], n_sep = a_numero(r["cantidad"])
    print(f"CANTIDAD con separador de miles: {n_sep}")
    clases = [norm(x) for x in a.clases.split(",")]
    n0 = len(r)
    r = r[(r["estado"].map(norm) == "activo") & r["clase"].map(norm).isin(clases)].copy()
    print(f"Filas: {n0}; activas y de clases elegidas: {len(r)}")
    alias = None
    if a.alias:
        al = pd.read_csv(a.alias, dtype=str)
        alias = {(norm(x), norm(y)): z.zfill(5) for x, y, z in zip(al["departamento"], al["municipio"], al["divipola"])}
    mapa = cruzar(r, dane, alias)
    r["divipola"] = [mapa[(d, m)][0] for d, m in zip(r["departamento"], r["municipio"])]
    rep = pd.DataFrame([(d, m, *v) for (d, m), v in mapa.items()], columns=["departamento", "municipio", "divipola", "tipo_cruce", "similitud"])
    rep.to_csv(os.path.join(a.out, "reporte_cruce_parque.csv"), index=False)
    print("Cruces:", rep["tipo_cruce"].value_counts().to_dict(), "| sin cruce:", int(r["divipola"].isna().sum()), "filas")
    out = r.dropna(subset=["divipola"]).groupby("divipola", as_index=False)["cantidad"].sum().rename(columns={"cantidad": "parque_liviano"})
    out.to_csv(os.path.join(a.out, "parque_liviano.csv"), index=False)
    print("parque_liviano.csv:", len(out), "municipios;", int(out["parque_liviano"].sum()), "vehículos")


if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    sp = ap.add_subparsers(dest="cmd", required=True)
    dd = sp.add_parser("dane")
    dd.add_argument("--xlsx", required=True), dd.add_argument("--anio", type=int, default=2026)
    dd.add_argument("--hoja", default="PobMunicipalxÁrea"), dd.add_argument("--out", default="salida_prep")
    e = sp.add_parser("explorar")
    e.add_argument("--runt", required=True), e.add_argument("--out", default="salida_prep")
    p = sp.add_parser("preparar")
    p.add_argument("--runt", required=True), p.add_argument("--dane", required=True)
    p.add_argument("--combustibles", required=True), p.add_argument("--clases", required=True)
    p.add_argument("--alias", help="CSV opcional departamento,municipio,divipola para cruces manuales")
    p.add_argument("--out", default="salida_prep")
    q = sp.add_parser("parque")
    q.add_argument("--runt", required=True), q.add_argument("--dane", required=True)
    q.add_argument("--clases", default="AUTOMOVIL,CAMIONETA,CAMPERO,MICROBUS")
    q.add_argument("--alias"), q.add_argument("--out", default="salida_prep")
    a = ap.parse_args()
    {"dane": dane_a_csv, "explorar": explorar, "preparar": preparar, "parque": parque}[a.cmd](a)

Writing prep_runt_datosgov.py


**2b. `eda_limpieza_datawizards.py`** perfila la calidad, limpia y calcula los indicadores y el análisis exploratorio (correlaciones, VIF, PCA, clusters).

In [3]:
%%writefile eda_limpieza_datawizards.py
#!/usr/bin/env python3
"""Anexo técnico Guía 4 - Datawizards.

Perfilado de calidad, limpieza y entendimiento inicial de datos para la
priorización de ciudades en carga pública de vehículos eléctricos.

Uso:
  python eda_limpieza_datawizards.py --demo --out salida_demo
  python eda_limpieza_datawizards.py --runt runt.csv --oferta oferta.csv \
      --poblacion dane.csv --out salida

Archivos de entrada esperados (columnas mínimas):
  runt.csv       salida de prep_runt_datosgov.py: divipola, anio, cantidad
                 (solo BEV de las clases elegidas, ya cruzado con DIVIPOLA)
  oferta.csv     fuente, nombre, lat, lon, conector (AC/DC/texto), potencia_kw, divipola
  dane.csv       divipola, municipio, poblacion, (opcionales: pib_pc, densidad_vial)

--demo genera DATOS SINTÉTICOS solo para probar el código. Sus resultados
NO deben copiarse al informe.
"""
import argparse
import os

import numpy as np
import pandas as pd
from rapidfuzz import fuzz
from scipy.cluster.hierarchy import fcluster, linkage
from scipy.stats import spearmanr
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler

COL_BBOX = dict(lat=(-4.3, 13.5), lon=(-79.1, -66.8))  # caja aproximada de Colombia
FUENTE_PRIORIDAD = {"SIEM": 0, "OCM": 1, "PORTAL": 2}
UMBRAL_FALTANTES = 0.10  # B2: <= 10 % antes de imputar o descartar
UMBRAL_DUP_NOMBRE = 90   # RapidFuzz >= 90
UMBRAL_DUP_METROS = 150
UMBRAL_CORR = 0.80       # R2
TOPE_CANTIDAD_FILA = 20000  # tope por fila (combinación clase-carrocería-modelo-año-municipio); ajustar tras ver la distribución real


# ----------------------------------------------------------------- datos demo
def generar_demo(seed=7):
    rng = np.random.default_rng(seed)
    n = 60
    cod = [f"{d:02d}{m:03d}" for d, m in zip(rng.integers(5, 99, n), rng.integers(1, 999, n))]
    cod = list(dict.fromkeys(cod))[:n]
    n = len(cod)
    pob = (rng.lognormal(11.5, 1.1, n)).astype(int)
    pob[0] = 7_900_000  # un caso tipo Bogotá, valor extremo legítimo
    dane = pd.DataFrame({"divipola": cod, "municipio": [f"Municipio {i}" for i in range(n)],
                         "poblacion": pob,
                         "pib_pc": rng.normal(20, 5, n).round(1),
                         "densidad_vial": (pob ** 0.3 * rng.normal(1, .1, n)).round(2)})
    dane.loc[rng.choice(n, 4, replace=False), "pib_pc"] = np.nan
    filas = []
    for i, c in enumerate(cod):
        base = max(1, pob[i] / 1e5 * 40)
        for k, a in enumerate(range(2018, 2027)):
            v = int(rng.poisson(base * (1 + k * .5)))
            filas.append((c, a, v))
    runt = pd.DataFrame(filas, columns=["divipola", "anio", "cantidad"])
    # defectos típicos
    runt["divipola"] = runt["divipola"].where(rng.random(len(runt)) > .03,
                                              runt["divipola"].str.lstrip("0"))
    runt.loc[rng.choice(len(runt), 6, replace=False), "cantidad"] = np.nan
    runt.loc[rng.choice(len(runt), 2, replace=False), "cantidad"] = -5
    runt.loc[rng.choice(len(runt), 2, replace=False), "cantidad"] = 99999
    runt = pd.concat([runt, runt.sample(5, random_state=1)])  # filas repetidas
    pts = []
    for fuente in ("SIEM", "OCM", "PORTAL"):
        for _ in range(110):
            i = rng.integers(0, n)
            pts.append((fuente, f"Estación {i}-{rng.integers(1, 6)} {fuente}",
                        rng.uniform(1, 11), rng.uniform(-77, -72), rng.choice(["AC", "DC", "CCS2", "Tipo 2", "CHAdeMO", None]),
                        rng.choice([7, 22, 50, 120, np.nan]), cod[i]))
    oferta = pd.DataFrame(pts, columns=["fuente", "nombre", "lat", "lon", "conector", "potencia_kw", "divipola"])
    dup = oferta.sample(40, random_state=3).copy()
    dup["fuente"] = "OCM"
    dup["nombre"] = dup["nombre"].str.replace("Estación", "Estacion ") + " "
    dup["lat"] += rng.normal(0, 3e-4, len(dup))
    oferta = pd.concat([oferta, dup], ignore_index=True)
    oferta.loc[rng.choice(len(oferta), 6, replace=False), ["lat", "lon"]] = [50.0, 8.0]  # fuera de Colombia
    return runt, oferta, dane


# ----------------------------------------------------------------- calidad
def std_divipola(s):
    return s.astype(str).str.replace(r"\.0$", "", regex=True).str.strip().str.zfill(5)


def perfil_calidad(df, nombre, reglas):
    """Devuelve filas del reporte: completitud, consistencia, claridad, formato."""
    out = []
    n = len(df)
    for c in df.columns:
        out.append((nombre, "Completitud", c, f"{df[c].isna().mean():.1%} vacíos", df[c].isna().sum()))
    out.append((nombre, "Completitud", "(filas)", "total", n))
    out.append((nombre, "Consistencia", "(duplicados exactos)", "filas repetidas", int(df.duplicated().sum())))
    for desc, mask in reglas.items():
        out.append((nombre, "Consistencia", desc, "incumplen", int(mask(df).sum())))
    return out


def reglas_runt(dane_cod):
    return {
        "cantidad negativa": lambda d: d["cantidad"] < 0,
        "cantidad > p99.9 (candidato a extremo)": lambda d: d["cantidad"] > d["cantidad"].quantile(.999),
        "divipola no tiene 5 dígitos": lambda d: d["divipola"].astype(str).str.len() != 5,
        "divipola fuera de catálogo DANE": lambda d: ~std_divipola(d["divipola"]).isin(dane_cod),
        "anio fuera de 2000-2030 o no numérico": lambda d: ~pd.to_numeric(d["anio"], errors="coerce").between(2000, 2030),
    }


def reglas_oferta(dane_cod):
    return {
        "lat fuera de Colombia": lambda d: ~d["lat"].between(*COL_BBOX["lat"]),
        "lon fuera de Colombia": lambda d: ~d["lon"].between(*COL_BBOX["lon"]),
        "potencia_kw no positiva o > 400": lambda d: (d["potencia_kw"] <= 0) | (d["potencia_kw"] > 400),
        "conector sin clasificar (ni AC ni DC)": lambda d: d["conector"].isna(),
        "divipola fuera de catálogo DANE": lambda d: ~std_divipola(d["divipola"]).isin(dane_cod),
    }


# ----------------------------------------------------------------- limpieza
def clasificar_conector(conector, potencia):
    """AC/DC según texto; si falta, según potencia (>=40 kW => DC). Devuelve (clase, origen)."""
    t = str(conector).upper()
    if t in ("AC", "TIPO 2", "TYPE 2", "J1772"):
        return "AC", "texto"
    if t in ("DC", "CCS2", "CCS", "CHADEMO", "GB/T"):
        return "DC", "texto"
    if pd.notna(potencia):
        return ("DC" if potencia >= 40 else "AC"), "potencia"
    return "SIN_CLASE", "ninguno"


def haversine_m(lat1, lon1, lat2, lon2):
    r = 6371000
    p1, p2 = np.radians(lat1), np.radians(lat2)
    a = np.sin((p2 - p1) / 2) ** 2 + np.cos(p1) * np.cos(p2) * np.sin(np.radians(lon2 - lon1) / 2) ** 2
    return 2 * r * np.arcsin(np.sqrt(a))


def deduplicar(of):
    """Marca duplicados entre fuentes: mismo municipio, <=150 m y nombre >=90. Conserva la fuente prioritaria."""
    of = of.copy()
    of["prio"] = of["fuente"].map(FUENTE_PRIORIDAD).fillna(9)
    of = of.sort_values(["prio"]).reset_index(drop=True)
    padre = list(range(len(of)))

    def raiz(i):
        while padre[i] != i:
            padre[i] = padre[padre[i]]
            i = padre[i]
        return i

    for _, g in of.groupby("divipola"):
        idx = g.index.to_list()
        for a_i, a in enumerate(idx):
            for b in idx[a_i + 1:]:
                d = haversine_m(of.at[a, "lat"], of.at[a, "lon"], of.at[b, "lat"], of.at[b, "lon"])
                s = fuzz.token_set_ratio(str(of.at[a, "nombre"]).lower().strip(), str(of.at[b, "nombre"]).lower().strip())
                if d <= UMBRAL_DUP_METROS and s >= UMBRAL_DUP_NOMBRE:
                    padre[raiz(b)] = raiz(a)
    of["grupo"] = [raiz(i) for i in range(len(of))]
    of["es_duplicado"] = of.duplicated("grupo", keep="first")
    return of.drop(columns="prio")


def limpiar(runt, oferta, dane, bitacora):
    cat = set(std_divipola(dane["divipola"]))
    # --- RUNT
    r = runt.copy()
    r["divipola"] = std_divipola(r["divipola"])
    n0 = len(r)
    r = r.drop_duplicates()
    bitacora.append(("RUNT", "Duplicados exactos eliminados", n0 - len(r)))
    neg = r["cantidad"] < 0
    bitacora.append(("RUNT", "Cantidades negativas -> NaN (no se imputan, se reportan)", int(neg.sum())))
    r.loc[neg, "cantidad"] = np.nan
    ext = r["cantidad"] > TOPE_CANTIDAD_FILA
    bitacora.append(("RUNT", f"Filas con cantidad > {TOPE_CANTIDAD_FILA:,} (imposible por fila) -> NaN para revisión manual", int(ext.sum())))
    r.loc[ext, "cantidad"] = np.nan
    fuera = ~r["divipola"].isin(cat)
    bitacora.append(("RUNT", "Filas con DIVIPOLA fuera del catálogo (excluidas)", int(fuera.sum())))
    r = r[~fuera]
    if oferta is None:
        return r, None, None
    # --- Oferta
    o = oferta.copy()
    o["divipola"] = std_divipola(o["divipola"])
    ok_geo = o["lat"].between(*COL_BBOX["lat"]) & o["lon"].between(*COL_BBOX["lon"])
    bitacora.append(("Oferta", "Puntos con coordenadas fuera de Colombia (excluidos)", int((~ok_geo).sum())))
    o = o[ok_geo].copy()
    cl = o.apply(lambda f: clasificar_conector(f["conector"], f["potencia_kw"]), axis=1)
    o["clase"] = [c[0] for c in cl]
    o["clase_origen"] = [c[1] for c in cl]
    bitacora.append(("Oferta", "Clase AC/DC inferida por potencia", int((o["clase_origen"] == "potencia").sum())))
    bitacora.append(("Oferta", "Sin clase AC/DC (se mantienen como SIN_CLASE, no se imputan)", int((o["clase"] == "SIN_CLASE").sum())))
    o = deduplicar(o)
    bitacora.append(("Oferta", "Puntos duplicados entre fuentes (RapidFuzz>=90 y <=150 m)", int(o["es_duplicado"].sum())))
    o_limpia = o[~o["es_duplicado"]].copy()
    return r, o_limpia, o


# ----------------------------------------------------------------- indicadores
def construir_indicadores(r, o, dane, anio_ref, parque=None):
    """r: divipola, anio, cantidad (solo BEV). anio_ref: último año COMPLETO (no usar el año en curso)."""
    d = dane.copy()
    d["divipola"] = std_divipola(d["divipola"])
    r = r.copy()
    r["anio"] = pd.to_numeric(r["anio"], errors="coerce")
    bev_total = r.groupby("divipola")["cantidad"].sum()
    bev_ref = r[r["anio"] == anio_ref].groupby("divipola")["cantidad"].sum()
    bev_prev = r[r["anio"] == anio_ref - 1].groupby("divipola")["cantidad"].sum()
    ind = d.set_index("divipola")
    # un municipio ausente del conjunto de eléctricos tiene 0 vehículos (no es dato faltante)
    ind["bev_acum"] = bev_total.reindex(ind.index).fillna(0)
    ind["sin_bev"] = ind["bev_acum"] == 0
    ind["matr_anio"] = bev_ref.reindex(ind.index).fillna(0)
    prev = bev_prev.reindex(ind.index).fillna(0)
    ind["crec_anual"] = np.where(prev > 0, ind["matr_anio"] / prev.replace(0, np.nan) - 1, np.nan)
    ind["bev_por_100k"] = ind["bev_acum"] / ind["poblacion"] * 1e5
    ind["matr_anio_por_100k"] = ind["matr_anio"] / ind["poblacion"] * 1e5
    if parque is not None:
        pq = parque.assign(divipola=std_divipola(parque["divipola"])).set_index("divipola")["parque_liviano"]
        ind["parque_liviano"] = pq.reindex(ind.index)
        ind["bev_pct_parque"] = np.where(ind["parque_liviano"] > 0, ind["bev_acum"] / ind["parque_liviano"] * 100, np.nan)
    if o is None:
        return ind, anio_ref
    cnt = o.groupby(["divipola", "clase"]).size().unstack(fill_value=0)
    for c in ("AC", "DC"):
        ind[f"con_{c}"] = cnt[c] if c in cnt else 0
        ind[f"con_{c}"] = ind[f"con_{c}"].fillna(0)
    ind["con_DC_por_1000bev"] = ind["con_DC"] / (ind["bev_acum"] / 1000).replace(0, np.nan)
    ind["con_AC_por_1000bev"] = ind["con_AC"] / (ind["bev_acum"] / 1000).replace(0, np.nan)
    return ind, anio_ref


def vif(X):
    X = (X - X.mean()) / X.std(ddof=0)
    out = {}
    for c in X.columns:
        y = X[c].values
        A = np.column_stack([np.ones(len(X)), X.drop(columns=c).values])
        beta, *_ = np.linalg.lstsq(A, y, rcond=None)
        r2 = 1 - ((y - A @ beta) ** 2).sum() / ((y - y.mean()) ** 2).sum()
        out[c] = 1 / (1 - r2) if r2 < 1 else np.inf
    return pd.Series(out)


def entendimiento(ind, outdir):
    n_total = len(ind)
    ind = ind[ind["bev_acum"] > 0]  # universo de exploración: municipios con al menos un eléctrico
    print(f"Universo de exploración: {len(ind)} de {n_total} municipios con al menos un eléctrico")
    cols = ["bev_por_100k", "bev_pct_parque", "matr_anio_por_100k", "crec_anual", "con_DC_por_1000bev", "con_AC_por_1000bev", "pib_pc", "densidad_vial", "dens_via_principal", "n_combustible", "n_comercio", "n_parqueaderos"]
    cols = [c for c in cols if c in ind.columns]
    res = {"n_universo": len(ind), "n_municipios_dane": n_total}
    # 5. estadísticas básicas + faltantes por indicador (B2)
    desc = ind[cols].describe(percentiles=[.05, .5, .95]).T
    desc["faltantes_%"] = ind[cols].isna().mean() * 100
    desc["cumple_B2"] = desc["faltantes_%"] <= UMBRAL_FALTANTES * 100
    desc.to_csv(f"{outdir}/eda_estadisticas_basicas.csv")
    res["desc"] = desc
    X = ind[cols].dropna()
    res["n_ciudades_completas"] = len(X)
    if len(X) < 8:
        return res
    # 3. reducción de dimensiones y 8. variables importantes (solo diagnóstico)
    corr = X.corr(method="spearman")
    corr.to_csv(f"{outdir}/eda_correlacion_spearman.csv")
    pares = [(a, b, round(corr.loc[a, b], 2)) for i, a in enumerate(cols) for b in cols[i + 1:] if abs(corr.loc[a, b]) > UMBRAL_CORR]
    res["pares_correlados"] = pares
    v = vif(X)
    v.to_csv(f"{outdir}/eda_vif.csv", header=["VIF"])
    res["vif"] = v
    pca = PCA().fit(StandardScaler().fit_transform(X))
    res["pca_var_acum"] = [float(x) for x in np.cumsum(pca.explained_variance_ratio_).round(3)]
    # 7/9. segmentación: clustering jerárquico (OA5), silueta >= 0,25
    Z = linkage(StandardScaler().fit_transform(X), "ward")
    mejor = None
    for k in range(2, 7):
        lab = fcluster(Z, k, "maxclust")
        if len(set(lab)) > 1:
            s = silhouette_score(StandardScaler().fit_transform(X), lab)
            if mejor is None or s > mejor[1]:
                mejor = (k, s, lab)
    if mejor:
        res["clusters"] = (mejor[0], round(mejor[1], 3), mejor[1] >= 0.25)
        pd.Series(mejor[2], index=X.index, name="cluster").to_csv(f"{outdir}/eda_clusters.csv")
    # Referencia B-1: ordenar por tamaño del parque; Spearman del índice ingenuo (suma de z) frente a B-1
    z = (X - X.mean()) / X.std(ddof=0)
    ingenuo = z.mean(axis=1)
    b1 = ind.loc[X.index, "bev_acum"]
    res["spearman_vs_B1"] = round(spearmanr(ingenuo, b1)[0], 3)
    return res


# ----------------------------------------------------------------- principal
def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--demo", action="store_true", help="usa datos sintéticos (solo prueba del código)")
    ap.add_argument("--runt"), ap.add_argument("--oferta"), ap.add_argument("--poblacion")
    ap.add_argument("--parque", help="parque_liviano.csv de prep_runt_datosgov.py parque (opcional)")
    ap.add_argument("--accesibilidad", help="accesibilidad_municipal.csv de osm_gpkg.py accesibilidad (opcional)")
    ap.add_argument("--anio-completo", type=int, default=None, help="último año completo de RUNT (p. ej. 2025)")
    ap.add_argument("--out", default="salida")
    a = ap.parse_args()
    os.makedirs(a.out, exist_ok=True)
    if a.demo:
        runt, oferta, dane = generar_demo()
    else:
        runt = pd.read_csv(a.runt, dtype={"divipola": str})
        oferta = pd.read_csv(a.oferta, dtype={"divipola": str}) if a.oferta else None  # sin oferta: solo demanda
        dane = pd.read_csv(a.poblacion, dtype={"divipola": str})
    cat = set(std_divipola(dane["divipola"]))

    # Parte I: calidad (antes de limpiar)
    rep = perfil_calidad(runt, "RUNT", reglas_runt(cat)) + perfil_calidad(dane, "DANE", {})
    if oferta is not None:
        rep += perfil_calidad(oferta, "Oferta", reglas_oferta(cat))
    pd.DataFrame(rep, columns=["fuente", "dimension", "campo_o_regla", "medida", "valor"]).to_csv(f"{a.out}/calidad_antes.csv", index=False)

    # Parte II: limpieza con bitácora
    bit = []
    r, o_limpia, o_marcada = limpiar(runt, oferta, dane, bit)
    pd.DataFrame(bit, columns=["fuente", "accion", "n"]).to_csv(f"{a.out}/bitacora_limpieza.csv", index=False)
    if o_marcada is not None:
        o_marcada.to_csv(f"{a.out}/oferta_con_marcas_duplicado.csv", index=False)  # para revisión manual de la muestra
    parque = pd.read_csv(a.parque, dtype={"divipola": str}) if a.parque else None

    # Parte III: entendimiento
    anio_ref = a.anio_completo or int(pd.to_numeric(r['anio'], errors='coerce').max()) - 1
    ind, corte = construir_indicadores(r, o_limpia, dane, anio_ref, parque)
    if a.accesibilidad:
        acc = pd.read_csv(a.accesibilidad, dtype={"divipola": str}).set_index("divipola")
        acc.index = std_divipola(acc.index.to_series()).values
        ind = ind.join(acc.drop(columns=["area_km2"], errors="ignore"))
    ind.to_csv(f"{a.out}/indicadores_por_municipio.csv")
    res = entendimiento(ind, a.out)

    with open(f"{a.out}/resumen.md", "w", encoding="utf-8") as f:
        f.write(f"# Resumen automático ({'DATOS SINTÉTICOS' if a.demo else 'datos reales'})\n\n")
        f.write(f"Último año completo usado para crecimiento anual: {corte} (el año en curso es parcial y no se compara)\n\n## Bitácora\n")
        for s, ac, n in bit:
            f.write(f"- {s}: {ac}: {n}\n")
        f.write(f"\n## Indicadores\nUniverso de exploración: {res.get('n_universo')} de {res.get('n_municipios_dane')} municipios (con al menos un eléctrico)\nMunicipios con todos los indicadores: {res.get('n_ciudades_completas')}\n\n")
        f.write(res["desc"][["count", "mean", "50%", "faltantes_%", "cumple_B2"]].round(2).to_markdown() + "\n")
        if "pares_correlados" in res:
            f.write(f"\nPares con |rho| > {UMBRAL_CORR}: {res['pares_correlados'] or 'ninguno'}\n")
            f.write(f"\nVIF:\n{res['vif'].round(2).to_markdown()}\n")
            f.write(f"\nVarianza acumulada PCA: {res['pca_var_acum']}\n")
            f.write(f"\nClusters (k, silueta, cumple >=0,25): {res.get('clusters')}\n")
            f.write(f"\nSpearman índice ingenuo vs B-1: {res['spearman_vs_B1']}\n")
    print(open(f"{a.out}/resumen.md", encoding="utf-8").read())


if __name__ == "__main__":
    main()

Writing eda_limpieza_datawizards.py


**2c. `osm_gpkg.py`** explora el GeoPackage de OpenStreetMap y calcula la accesibilidad por municipio.

In [4]:
%%writefile osm_gpkg.py
#!/usr/bin/env python3
"""Revisión y accesibilidad con OpenStreetMap (Geofabrik colombia-YYMMDD-free.gpkg).

1) Explorar (no necesita límites municipales):
   python osm_gpkg.py explorar --gpkg colombia-261108-free.gpkg --out osm
   -> capas_gpkg.csv (capas, tipo de geometría, filas, columnas), fclass_<capa>.csv (tipos por capa)
      y puntos_carga_osm.csv (candidatos a estación de carga: por tipo o por nombre).

2) Accesibilidad por municipio (necesita los límites municipales del DANE, MGN):
   python osm_gpkg.py accesibilidad --gpkg colombia-261108-free.gpkg --mgn MGN_MPIO.shp --out osm
   -> accesibilidad_municipal.csv: divipola, area_km2, km_via_principal, dens_via_principal,
      n_estaciones_combustible, n_parqueaderos, n_comercio (según las clases disponibles).

Los nombres de capa y de clase (fclass) cambian de una versión a otra: el script los busca por
palabra clave y registra lo que encuentra. Revise capas_gpkg.csv y fclass_*.csv antes de confiar.
"""
import argparse
import os
import re

import geopandas as gpd
import numpy as np
import pandas as pd
import pyogrio

CRS_M = "EPSG:9377"  # MAGNA-SIRGAS Origen Nacional (metros)
VIAS_PRINCIPALES = ["motorway", "motorway_link", "trunk", "trunk_link", "primary", "primary_link", "secondary", "secondary_link"]
PATRON_CARGA = re.compile(r"charg|carga|electrolin|eléctric|electric|voltex|ev\s?point|wallbox", re.I)


def capas(gpkg):
    info = pyogrio.list_layers(gpkg)
    return [(str(n), str(g)) for n, g in info]


def buscar_capa(lista, clave, poligono=False):
    """Capa cuyo nombre contiene la clave; '_a_' en el nombre indica polígonos en Geofabrik."""
    c = [n for n, _ in lista if re.search(rf"(^|_){clave}(_|$)", n)]
    c = [n for n in c if (("_a_" in n or n.endswith("_a")) == poligono)]
    return c[0] if c else None


def explorar(a):
    os.makedirs(a.out, exist_ok=True)
    lista = capas(a.gpkg)
    filas = []
    for n, g in lista:
        info = pyogrio.read_info(a.gpkg, layer=n)
        filas.append((n, g, info["features"], info["crs"], ", ".join(info["fields"])))
    df = pd.DataFrame(filas, columns=["capa", "geometria", "filas", "crs", "columnas"])
    df.to_csv(os.path.join(a.out, "capas_gpkg.csv"), index=False)
    print(df[["capa", "geometria", "filas"]].to_string(index=False))

    carga = []
    for n, _ in lista:
        cols = [c for c in pyogrio.read_info(a.gpkg, layer=n)["fields"]]
        if "fclass" not in cols:
            continue
        t = pyogrio.read_dataframe(a.gpkg, layer=n, read_geometry=False, columns=[c for c in ("fclass", "name") if c in cols])
        t["fclass"].value_counts().rename_axis("fclass").reset_index(name="n").to_csv(os.path.join(a.out, f"fclass_{n}.csv"), index=False)
        if re.search(r"pois|transport|traffic", n):
            m = t["fclass"].astype(str).str.contains(PATRON_CARGA) | (t["name"].astype(str).str.contains(PATRON_CARGA) if "name" in t else False)
            if m.any():
                g = pyogrio.read_dataframe(a.gpkg, layer=n)
                g = g[m.values].copy()
                g["capa"] = n
                carga.append(g)
    if carga:
        g = pd.concat(carga)
        g = gpd.GeoDataFrame(g, geometry="geometry", crs=carga[0].crs)
        cen = g.to_crs(CRS_M).geometry.centroid.to_crs("EPSG:4326")
        out = pd.DataFrame({"fuente": "OSM", "nombre": g.get("name"), "fclass": g.get("fclass"), "capa": g["capa"],
                            "lat": cen.y.values, "lon": cen.x.values})
        out.to_csv(os.path.join(a.out, "puntos_carga_osm.csv"), index=False)
        print(f"\nPuntos candidatos a estación de carga en OSM: {len(out)} (revisar a mano; sirven como contraste, no como oferta oficial)")
    else:
        print("\nNo se encontraron estaciones de carga por tipo ni por nombre en las capas de puntos de interés; para ellas use el .pbf (amenity=charging_station).")


def leer_mgn(ruta, columna):
    m = gpd.read_file(ruta)
    if columna is None:
        cand = [c for c in m.columns if "CDPMP" in c.upper()] or [c for c in m.columns if c.upper() in ("MPIO_CCNCT", "COD_MPIO", "DIVIPOLA")]
        if not cand:
            raise SystemExit(f"No encuentro la columna con el código del municipio. Columnas: {list(m.columns)}. Use --mgn-codigo")
        columna = cand[0]
    m = m[[columna, "geometry"]].rename(columns={columna: "divipola"})
    m["divipola"] = m["divipola"].astype(str).str.replace(r"\.0$", "", regex=True).str.zfill(5)
    m = m.to_crs(CRS_M)
    m["area_km2"] = m.geometry.area / 1e6
    return m


def km_por_municipio(lineas, mgn):
    """Longitud (km) de las líneas dentro de cada municipio (recorta por polígono)."""
    out = {}
    sidx = lineas.sindex
    for fila in mgn.itertuples():
        pos = list(sidx.query(fila.geometry, predicate="intersects"))
        if pos:
            out[fila.divipola] = lineas.iloc[pos].intersection(fila.geometry).length.sum() / 1000
    return pd.Series(out, name="km")


def conteo_por_municipio(puntos, mgn):
    j = gpd.sjoin(puntos.to_crs(CRS_M), mgn[["divipola", "geometry"]], predicate="within", how="inner")
    return j.groupby("divipola").size()


def accesibilidad(a):
    os.makedirs(a.out, exist_ok=True)
    lista = capas(a.gpkg)
    mgn = leer_mgn(a.mgn, a.mgn_codigo)
    res = mgn[["divipola", "area_km2"]].drop_duplicates("divipola").set_index("divipola")

    vias = buscar_capa(lista, "roads")
    print("Capa de vías:", vias)
    v = pyogrio.read_dataframe(a.gpkg, layer=vias, columns=["fclass"])
    v = v[v["fclass"].isin(a.clases_via.split(","))].to_crs(CRS_M)
    print("Tramos de vía principal:", len(v), "|", v["fclass"].value_counts().to_dict())
    res["km_via_principal"] = km_por_municipio(v, mgn.drop_duplicates("divipola").set_index("divipola", drop=False))
    res["km_via_principal"] = res["km_via_principal"].fillna(0)
    res["dens_via_principal"] = res["km_via_principal"] / res["area_km2"]

    # Geofabrik guarda 'fuel' y 'parking' en la capa traffic y los comercios en pois: se busca en las dos
    for etiqueta, claves, clases in (("n_combustible", ("traffic", "pois"), ["fuel"]),
                                     ("n_comercio", ("pois",), ["mall", "supermarket"]),
                                     ("n_parqueaderos", ("traffic", "pois"), ["parking", "parking_multistorey", "parking_underground"])):
        partes = []
        for clave in claves:
            capa = buscar_capa(lista, clave, False)
            if not capa:
                continue
            p = pyogrio.read_dataframe(a.gpkg, layer=capa, columns=["fclass"])
            p = p[p["fclass"].isin(clases)]
            if len(p):
                partes.append(p)
                print(f"{etiqueta}: {len(p)} puntos en {capa} (clases {clases})")
        if not partes:
            print(f"{etiqueta}: sin puntos en {claves}; se omite")
            continue
        p = pd.concat(partes)
        res[etiqueta] = conteo_por_municipio(p, mgn).reindex(res.index).fillna(0)

    res.reset_index().to_csv(os.path.join(a.out, "accesibilidad_municipal.csv"), index=False)
    print("accesibilidad_municipal.csv:", len(res), "municipios")


if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    sp = ap.add_subparsers(dest="cmd", required=True)
    e = sp.add_parser("explorar")
    e.add_argument("--gpkg", required=True), e.add_argument("--out", default="osm")
    q = sp.add_parser("accesibilidad")
    q.add_argument("--gpkg", required=True), q.add_argument("--mgn", required=True)
    q.add_argument("--mgn-codigo", default=None, help="columna con el código DIVIPOLA en el MGN (por defecto busca CDPMP)")
    q.add_argument("--clases-via", default=",".join(VIAS_PRINCIPALES))
    q.add_argument("--out", default="osm")
    a = ap.parse_args()
    {"explorar": explorar, "accesibilidad": accesibilidad}[a.cmd](a)

Writing osm_gpkg.py


**2d. `mgn_desde_osm.py`** construye los polígonos municipales con códigos DIVIPOLA a partir de OSM. Sustituye al MGN del DANE mientras no se tenga.

In [5]:
%%writefile mgn_desde_osm.py
#!/usr/bin/env python3
"""Plan B del MGN: arma límites municipales con la capa adminareas del gpkg de Geofabrik.

OSM Colombia usa admin_level4 = departamento y admin_level6 = municipio. La capa no trae el
código DIVIPOLA, así que se asigna cruzando departamento + nombre del municipio con el DANE.

Uso:
  python mgn_desde_osm.py --gpkg colombia.gpkg --dane prep/dane_municipios.csv --out osm
Salidas: osm/mgn_osm.gpkg (columna divipola, para usar con --mgn y --mgn-codigo divipola)
         osm/reporte_cruce_osm.csv (cruce de cada polígono) y osm/dane_sin_poligono.csv
"""
import argparse
import os
import re
import unicodedata

import geopandas as gpd
import pandas as pd
from rapidfuzz import fuzz, process

UMBRAL_DEP = 85
UMBRAL_MUN = 90
UMBRAL_SUBCONJUNTO = 95


def norm(t):
    t = unicodedata.normalize("NFKD", str(t)).encode("ascii", "ignore").decode().lower()
    t = re.sub(r"[^a-z0-9 ]", " ", t)
    t = re.sub(r"\b(distrito|capital|dc|d c|municipio de)\b", " ", t)
    return re.sub(r"\s+", " ", t).strip()


def sin_parentesis(t):
    return re.sub(r"\s*\(.*?\)\s*", " ", str(t)).strip()


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--gpkg", required=True), ap.add_argument("--dane", required=True)
    ap.add_argument("--capa", default="gis_osm_adminareas_a_free"), ap.add_argument("--out", default="osm")
    ap.add_argument("--alias", help="CSV opcional departamento_osm,municipio_osm,divipola")
    a = ap.parse_args()
    os.makedirs(a.out, exist_ok=True)

    adm = gpd.read_file(a.gpkg, layer=a.capa)
    dep = adm[adm["fclass"] == "admin_level4"][["name", "geometry"]].rename(columns={"name": "dep_osm"})
    mun = adm[adm["fclass"] == "admin_level6"][["osm_id", "name", "geometry"]].copy()
    print(f"Departamentos (nivel 4): {len(dep)} | municipios (nivel 6): {len(mun)}")

    # departamento de cada municipio: el polígono de nivel 4 que contiene un punto interior del municipio
    pts = mun.copy()
    pts["geometry"] = mun.geometry.representative_point()
    j = gpd.sjoin(pts, dep, predicate="within", how="left").drop_duplicates("osm_id")
    mun = mun.merge(j[["osm_id", "dep_osm"]], on="osm_id", how="left")

    dane = pd.read_csv(a.dane, dtype={"divipola": str})
    dane["k_dep"], dane["k_mun"] = dane["departamento"].map(norm), dane["municipio"].map(norm)
    por_dep = {k: g for k, g in dane.groupby("k_dep")}
    deps = list(por_dep)

    alias = {}
    if a.alias:  # CSV: departamento_osm, municipio_osm, divipola
        al = pd.read_csv(a.alias, dtype=str)
        alias = {(norm(x), norm(sin_parentesis(y))): z.zfill(5) for x, y, z in zip(al["departamento_osm"], al["municipio_osm"], al["divipola"])}

    filas = []
    for r in mun.itertuples():
        kd = norm(r.dep_osm) if isinstance(r.dep_osm, str) else ""
        # departamento: coincidencia exacta; si no, similitud total (NO token_set: "Santander" ~ "Norte de Santander")
        d = (kd, 100) if kd in por_dep else (process.extractOne(kd, deps, scorer=fuzz.ratio) if kd else None)
        divipola, tipo, sim = None, "sin_cruce", 0
        km = norm(sin_parentesis(r.name))
        if (kd, km) in alias:
            divipola, tipo, sim = alias[(kd, km)], "alias", 100
        elif d and d[1] >= UMBRAL_DEP:
            g = por_dep[d[0]]
            ex = g[g["k_mun"] == km]
            if len(ex) == 1:
                divipola, tipo, sim = ex.iloc[0]["divipola"], "exacto", 100
            else:
                ts = [fuzz.token_set_ratio(km, x) for x in g["k_mun"]]
                if ts and max(ts) >= UMBRAL_SUBCONJUNTO and ts.count(max(ts)) == 1:  # p. ej. "Cúcuta" ~ "San José de Cúcuta"
                    divipola, tipo, sim = g.iloc[ts.index(max(ts))]["divipola"], "subconjunto", max(ts)
                else:
                    m = process.extractOne(km, list(g["k_mun"]), scorer=fuzz.ratio)
                    if m and m[1] >= UMBRAL_MUN:
                        divipola, tipo, sim = g.iloc[m[2]]["divipola"], "difuso", round(m[1], 1)
        filas.append((r.osm_id, r.name, r.dep_osm, divipola, tipo, sim))
    rep = pd.DataFrame(filas, columns=["osm_id", "municipio_osm", "departamento_osm", "divipola", "tipo_cruce", "similitud"])
    rep.to_csv(os.path.join(a.out, "reporte_cruce_osm.csv"), index=False)
    print("Cruce:", rep["tipo_cruce"].value_counts().to_dict())

    mun = mun.merge(rep[["osm_id", "divipola"]], on="osm_id")
    ok = mun.dropna(subset=["divipola"])
    # Bogotá, D.C. es departamento (nivel 4) en OSM y no siempre tiene polígono de nivel 6
    if "11001" not in set(ok["divipola"]):
        bog = adm[(adm["fclass"] == "admin_level4") & adm["name"].map(norm).str.startswith("bogota")]
        if len(bog):
            ok = pd.concat([ok, pd.DataFrame({"osm_id": [bog.iloc[0]["osm_id"]], "name": ["Bogotá (nivel 4)"], "geometry": [bog.iloc[0].geometry], "dep_osm": ["Bogotá"], "divipola": ["11001"]})])
            print("Bogotá D.C. tomada del polígono de nivel 4")
    dup = ok["divipola"].duplicated(keep=False).sum()
    if dup:
        print(f"{dup} polígonos comparten código: se unen en uno solo por municipio")
    out = ok.dissolve(by="divipola", as_index=False)[["divipola", "geometry"]]
    out.to_file(os.path.join(a.out, "mgn_osm.gpkg"), layer="municipios", driver="GPKG")
    falta = dane[~dane["divipola"].isin(out["divipola"])][["divipola", "departamento", "municipio"]]
    falta.to_csv(os.path.join(a.out, "dane_sin_poligono.csv"), index=False)
    print(f"Municipios DANE con polígono: {len(out)} de {len(dane)}; sin polígono: {len(falta)} (ver dane_sin_poligono.csv)")


if __name__ == "__main__":
    main()

Writing mgn_desde_osm.py


**2e. Tablas de alias** (correcciones manuales de nombres).
- `alias_municipios.csv`: municipios del RUNT cuyo nombre no coincide con el DANE.
- `alias_osm.csv`: municipios de OSM cuyo nombre no coincide con el DANE. Aquí se agregan los que queden sin polígono en el paso 7.

In [6]:
%%writefile alias_municipios.csv
departamento,municipio,divipola
Norte de Santander,CUCUTA,54001
Bolivar,CARTAGENA,13001
Cundinamarca,UBATE,25843
Cauca,PIENDAMO,19548
Narino,TUMACO,52835
Sucre,SINCE,70742

Writing alias_municipios.csv


In [7]:
%%writefile alias_osm.csv
departamento_osm,municipio_osm,divipola
Bolívar,Santa Cruz de Mompox,13468
Sucre,Tolú,70820

Writing alias_osm.csv


## 3. Localizar los archivos de datos
Busca los tres archivos en cualquier carpeta de `/content`, sin importar dónde se hayan subido. Si falta alguno, o hay dos que coinciden, se detiene con un mensaje claro.

In [8]:
import glob, os

def buscar(patron):
    r = [f for f in glob.glob(f"/content/**/{patron}", recursive=True) if os.path.isfile(f)]
    assert len(r) == 1, f"Esperaba 1 archivo con el patrón {patron!r} y encontré {r}. Súbalo o quite los duplicados."
    return r[0]

EV     = buscar("*RUNT_2.0*.csv")      # eléctricos e híbridos matriculados
PARQUE = buscar("*PARQUE*.csv")        # parque automotor
DANE   = buscar("PPED*.xlsx")          # población DANE
print(EV, PARQUE, DANE, sep="\n")

/content/sample_data/VEHÍCULOS_ELÉCTRICOS_E_HÍBRIDOS_MATRICULADOS_EN_COLOMBIA_-_RUNT_2.0_20261109.csv
/content/sample_data/CRECIMIENTO_DEL_PARQUE_AUTOMOTOR_RUNT2.0_20261109.csv
/content/sample_data/PPED-AreaMun-2018-2042_VP.xlsx


## 4. Preparar el RUNT y el DANE
**4a. Población municipal** (hoja `PobMunicipalxÁrea`, área "Total", año 2026).
**4b. Explorar el RUNT:** lista los valores de combustible y clase, y los años. Revise la salida antes de fijar filtros.

In [9]:
!python prep_runt_datosgov.py dane --xlsx "{DANE}" --anio 2026 --out prep
!python prep_runt_datosgov.py explorar --runt "{EV}" --out prep

dane_municipios.csv: 1123 municipios, población 2026: 53,399,171
Filas: 10748 | CANTIDAD con separador de miles: 53 | CANTIDAD no numérica: 0
Vacíos por columna:
 combustible     0
modelo          0
anio            0
clase           0
carroceria      1
municipio       0
departamento    0
cantidad        0
dtype: int64
combustible
DIES ELEC      3146
ELECTRICO     89900
GASO ELEC    239851
HIDROGENO         4
Name: cantidad, dtype: int64
Revise los CSV de valores antes de preparar.


**4c. Filtrar y cruzar.** Los filtros son una **propuesta**: solo eléctricos puros (`ELECTRICO`) y vehículos livianos. Si el equipo decide otra cosa (por ejemplo incluir híbridos o buses), cámbielos aquí y vuelva a correr desde esta celda.
Salidas: `runt_anual.csv`, `parque_liviano.csv`, `reporte_cruce_municipios.csv` (qué municipios cruzaron y cómo) y `calidad_runt_crudo.csv`.

In [10]:
# AJUSTAR: combustibles y clases según la decisión del equipo
!python prep_runt_datosgov.py preparar --runt "{EV}" --dane prep/dane_municipios.csv --combustibles ELECTRICO --clases "AUTOMOVIL,CAMIONETA,CAMPERO" --alias alias_municipios.csv --out prep
!python prep_runt_datosgov.py parque --runt "{PARQUE}" --dane prep/dane_municipios.csv --clases "AUTOMOVIL,CAMIONETA,CAMPERO" --alias alias_municipios.csv --out prep

CANTIDAD con separador de miles corregidas: 53
Vacíos en el archivo crudo: {'carroceria': 1}
Filas del conjunto: 10748; tras filtrar combustible y clase: 2141
Filas sin cruce con DANE: 0 (0 vehículos). Revise reporte_cruce_municipios.csv
Cruces difusos para validar a mano: 1
Filas filtradas con cantidad vacía (no suman): 0
runt_anual.csv: 572 filas
CANTIDAD con separador de miles: 0
Filas: 192196; activas y de clases elegidas: 74690
Cruces: {'exacto': 281, 'alias': 6, 'difuso': 2} | sin cruce: 0 filas
parque_liviano.csv: 289 municipios; 7010273 vehículos


## 5. Calidad, limpieza y EDA de demanda
Calcula por municipio: eléctricos por 100 mil habitantes, % del parque, matrículas del último año completo por 100 mil y crecimiento anual. Aplica las reglas de la Guía 4 (faltantes ≤10 %, correlación >0,8, silueta ≥0,25) y deja una bitácora de limpieza en `prep/eda/resumen.md`.
Esta corrida usa **solo demanda**; la accesibilidad se agrega en el paso 9.

In [11]:
# AJUSTAR: --anio-completo es el último año con datos completos en el RUNT
!python eda_limpieza_datawizards.py --runt prep/runt_anual.csv --poblacion prep/dane_municipios.csv --parque prep/parque_liviano.csv --anio-completo 2025 --out prep/eda

Universo de exploración: 154 de 1123 municipios con al menos un eléctrico
# Resumen automático (datos reales)

Último año completo usado para crecimiento anual: 2025 (el año en curso es parcial y no se compara)

## Bitácora
- RUNT: Duplicados exactos eliminados: 0
- RUNT: Cantidades negativas -> NaN (no se imputan, se reportan): 0
- RUNT: Filas con cantidad > 20,000 (imposible por fila) -> NaN para revisión manual: 0
- RUNT: Filas con DIVIPOLA fuera del catálogo (excluidas): 0

## Indicadores
Universo de exploración: 154 de 1123 municipios (con al menos un eléctrico)
Municipios con todos los indicadores: 79

|                    |   count |   mean |   50% |   faltantes_% | cumple_B2   |
|:-------------------|--------:|-------:|------:|--------------:|:------------|
| bev_por_100k       |     154 |  84.44 | 11.77 |           0   | True        |
| bev_pct_parque     |     154 |   0.55 |  0.12 |           0   | True        |
| matr_anio_por_100k |     154 |  25.2  |  3.05 |           0   

In [12]:
import pandas as pd
ind = pd.read_csv("prep/eda/indicadores_por_municipio.csv", dtype={"divipola": str})
ind.sort_values("bev_acum", ascending=False).head(15)   # los 15 municipios con más eléctricos

,divipola,departamento,municipio,poblacion,bev_acum,sin_bev,matr_anio,crec_anual,bev_por_100k,matr_anio_por_100k,parque_liviano,bev_pct_parque
148,11001,"Bogotá, D.C.","Bogotá, D.C.",7945996,35952.0,False,10296.0,0.990334,452.454293,129.574694,2238115.0,1.606352
0,05001,Antioquia,Medellín,2526795,9045.0,False,2072.0,1.523752,357.963349,82.001112,371619.0,2.433944
46,05266,Antioquia,Envigado,268381,4831.0,False,1276.0,1.328467,1800.052910,475.443493,358750.0,1.346620
1006,76001,Valle del Cauca,Cali,2269983,4231.0,False,1667.0,1.835034,186.389061,73.436673,563925.0,0.750277
486,25286,Cundinamarca,Funza,125880,1411.0,False,393.0,0.369338,1120.908802,312.202097,156462.0,0.901816
86,05631,Antioquia,Sabaneta,109260,1033.0,False,316.0,1.796460,945.451217,289.218378,132357.0,0.780465
473,25175,Cundinamarca,Chía,177550,958.0,False,279.0,0.007220,539.566319,157.138834,78080.0,1.226947
832,66001,Risaralda,Pereira,487820,768.0,False,246.0,0.344262,157.435120,50.428437,122669.0,0.626075
318,17001,Caldas,Manizales,475690,697.0,False,350.0,6.142857,146.523997,73.577330,139902.0,0.498206
125,08001,Atlántico,Barranquilla,1275854,653.0,False,267.0,2.869565,51.181405,20.927159,172323.0,0.378940


## 6. OpenStreetMap: descarga y exploración
Descarga el GeoPackage de Geofabrik (~320 MB; tarda unos minutos) y lo valida. Si ya lo descargó, la celda no vuelve a bajarlo.
`explorar` lista las capas y los tipos (`fclass`) en `osm/capas_gpkg.csv` y `osm/fclass_*.csv`.

In [13]:
import glob, os
if not glob.glob("colombia*.gpkg"):
    !wget -c -q --show-progress -O colombia-latest-free.gpkg.zip https://download.geofabrik.de/south-america/colombia-latest-free.gpkg.zip
    !unzip -tq colombia-latest-free.gpkg.zip      # verifica que el zip no esté truncado
    !unzip -o -q colombia-latest-free.gpkg.zip -d .
GPKG = glob.glob("colombia*.gpkg")[0]
print("GeoPackage:", GPKG, f"({os.path.getsize(GPKG)/1e6:.0f} MB)")
!python osm_gpkg.py explorar --gpkg "{GPKG}" --out osm

colombia-latest-fre 100%[===================>] 766.14M  15.3MB/s    in 51s     
No errors detected in compressed data of colombia-latest-free.gpkg.zip.
GeoPackage: colombia.gpkg (1615 MB)
                          capa    geometria   filas
          gis_osm_traffic_free        Point   77251
           gis_osm_places_free        Point   49949
        gis_osm_transport_free        Point    8964
          gis_osm_natural_free        Point  135892
             gis_osm_pois_free        Point  126777
             gis_osm_pofw_free        Point    1620
            gis_osm_roads_free   LineString 1157263
        gis_osm_waterways_free   LineString   83478
          gis_osm_water_a_free MultiPolygon   51896
         gis_osm_places_a_free MultiPolygon    1422
gis_osm_protected_areas_a_free MultiPolygon    1433
        gis_osm_landuse_a_free MultiPolygon  164924
         gis_osm_railways_free   LineString    1019
           gis_osm_pois_a_free MultiPolygon   84294
      gis_osm_buildings_a_free M

**Estaciones de carga en OSM.** El archivo gratuito de Geofabrik **no trae** una clase de estaciones de carga. La búsqueda por palabras produce falsos positivos (ferreterías `doityourself`, la escuela "Charguayaco"). Por eso esta lista **no sirve como oferta de carga**; la oferta debe venir de Open Charge Map y de los operadores. La celda siguiente solo permite revisarlo.

In [14]:
c = pd.read_csv("osm/puntos_carga_osm.csv")
print(c["fclass"].value_counts().head(10))
estricto = c[c["nombre"].astype(str).str.contains(
    r"charg|electrolin|ev\s?point|voltex|wallbox|estaci[oó]n de carga|punto de carga|cargador", case=False, regex=True)]
print("Candidatas con nombre claro de carga de vehículos:", len(estricto))
estricto[["nombre", "fclass", "lat", "lon"]].head(40)

fclass
doityourself    179
viewpoint        12
school           10
stationery        4
bicycle_shop      3
parking           3
apron             3
tourist_info      3
restaurant        3
bus_stop          2
Name: count, dtype: int64
Candidatas con nombre claro de carga de vehículos: 7


,nombre,fclass,lat,lon
3,Escuela Paraiso Charguayaco,school,1.782117,-76.052554
4,Escuela Charguayaco,school,1.782317,-76.045744
5,Escuela Charguayaco,school,1.639167,-77.262937
6,Escuela Charguayaco,school,1.639127,-77.262917
186,Escuela Nueva Charguayaco,school,1.649167,-77.273333
188,Escuela Nueva Charguayaco,school,1.258760,-77.330809
233,Escuela Charguayaco,school,1.811879,-76.015733


## 7. Límites municipales desde OSM (plan B del MGN)
OSM trae polígonos de municipios (`admin_level6`) y departamentos (`admin_level4`), pero sin código DIVIPOLA. El script lo asigna cruzando departamento y nombre con el DANE. Cuando se tenga el MGN oficial del DANE, se reemplaza este paso.

In [15]:
!python mgn_desde_osm.py --gpkg "{GPKG}" --dane prep/dane_municipios.csv --alias alias_osm.csv --out osm

Departamentos (nivel 4): 33 | municipios (nivel 6): 1125
Cruce: {'exacto': 1068, 'subconjunto': 41, 'sin_cruce': 10, 'difuso': 4, 'alias': 2}
2 polígonos comparten código: se unen en uno solo por municipio
Municipios DANE con polígono: 1114 de 1123; sin polígono: 9 (ver dane_sin_poligono.csv)


**Diagnóstico del cruce:** qué polígonos no cruzaron y si afecta a municipios que tienen eléctricos. Lo que importa es que los municipios con eléctricos tengan polígono. Para los que falten, agregue una fila a `alias_osm.csv` (departamento_osm, municipio_osm, divipola) y repita el paso 7.

In [16]:
rep = pd.read_csv("osm/reporte_cruce_osm.csv", dtype=str)
print("Tipos de cruce:", rep["tipo_cruce"].value_counts().to_dict())
print("\nPolígonos de OSM sin cruce:")
print(rep[rep["tipo_cruce"] == "sin_cruce"][["municipio_osm", "departamento_osm"]].to_string(index=False))

ind = pd.read_csv("prep/eda/indicadores_por_municipio.csv", dtype={"divipola": str})
falta = pd.read_csv("osm/dane_sin_poligono.csv", dtype=str)
relev = ind[(ind["bev_acum"] > 0) & ind["divipola"].isin(falta["divipola"])]
print(f"\nMunicipios con eléctricos sin polígono: {len(relev)} de {int((ind['bev_acum'] > 0).sum())}")
print(relev[["divipola", "municipio", "departamento", "bev_acum"]].sort_values("bev_acum", ascending=False).to_string(index=False))

# Cruces aproximados: confirme a ojo que no sean errores
print("\nCruces aproximados (difuso/subconjunto), muestra:")
print(rep[rep["tipo_cruce"].isin(["difuso", "subconjunto"])][["municipio_osm", "departamento_osm", "divipola", "tipo_cruce"]].head(15).to_string(index=False))

Tipos de cruce: {'exacto': 1068, 'subconjunto': 41, 'sin_cruce': 10, 'difuso': 4, 'alias': 2}

Polígonos de OSM sin cruce:
    municipio_osm departamento_osm
  Área en litigio            Cauca
       Las Ánimas            Chocó
   Morichal Nuevo          Guainía
        El Atrato            Chocó
       El Vesubio        Antioquia
          Obregón        Antioquia
        Cucurucho        Antioquia
        La Zuleja        Antioquia
      Brechoncito        Antioquia
Penderisco Arriba        Antioquia

Municipios con eléctricos sin polígono: 2 de 154
divipola  municipio                                             departamento  bev_acum
   17001  Manizales                                                   Caldas     697.0
   88001 San Andrés Archipiélago de San Andrés, Providencia y Santa Catalina      10.0

Cruces aproximados (difuso/subconjunto), muestra:
         municipio_osm   departamento_osm divipola  tipo_cruce
                 Sincé              Sucre    70742 subconjunto
    

## 8. Accesibilidad por municipio
Con los polígonos del paso 7 calcula, por municipio: kilómetros de vía principal (autopista, troncal, primaria y secundaria), densidad de vía principal (km/km²) y conteos de estaciones de combustible, comercio (centros comerciales y supermercados) y parqueaderos.
**Limitación:** los polígonos de OSM pueden incluir mar en municipios costeros y eso infla el área. En esos municipios la densidad sale más baja de lo real.

In [17]:
!python osm_gpkg.py accesibilidad --gpkg "{GPKG}" --mgn osm/mgn_osm.gpkg --mgn-codigo divipola --out osm

Capa de vías: gis_osm_roads_free
Tramos de vía principal: 64170 | {'secondary': 24146, 'primary': 16733, 'trunk': 15498, 'trunk_link': 3905, 'primary_link': 2236, 'secondary_link': 1652}
n_combustible: 653 puntos en gis_osm_traffic_free (clases ['fuel'])
n_comercio: 4160 puntos en gis_osm_pois_free (clases ['mall', 'supermarket'])
n_parqueaderos: 1318 puntos en gis_osm_traffic_free (clases ['parking', 'parking_multistorey', 'parking_underground'])
accesibilidad_municipal.csv: 1114 municipios


## 9. EDA final: demanda + accesibilidad
Repite el análisis del paso 5 sumando los indicadores de accesibilidad.

In [18]:
!python eda_limpieza_datawizards.py --runt prep/runt_anual.csv --poblacion prep/dane_municipios.csv --parque prep/parque_liviano.csv --accesibilidad osm/accesibilidad_municipal.csv --anio-completo 2025 --out prep/eda_acc
print(open("prep/eda_acc/resumen.md", encoding="utf-8").read())

Universo de exploración: 154 de 1123 municipios con al menos un eléctrico
# Resumen automático (datos reales)

Último año completo usado para crecimiento anual: 2025 (el año en curso es parcial y no se compara)

## Bitácora
- RUNT: Duplicados exactos eliminados: 0
- RUNT: Cantidades negativas -> NaN (no se imputan, se reportan): 0
- RUNT: Filas con cantidad > 20,000 (imposible por fila) -> NaN para revisión manual: 0
- RUNT: Filas con DIVIPOLA fuera del catálogo (excluidas): 0

## Indicadores
Universo de exploración: 154 de 1123 municipios (con al menos un eléctrico)
Municipios con todos los indicadores: 77

|                    |   count |   mean |   50% |   faltantes_% | cumple_B2   |
|:-------------------|--------:|-------:|------:|--------------:|:------------|
| bev_por_100k       |     154 |  84.44 | 11.77 |           0   | True        |
| bev_pct_parque     |     154 |   0.55 |  0.12 |           0   | True        |
| matr_anio_por_100k |     154 |  25.2  |  3.05 |           0   

## 10. Descargar los resultados
El zip contiene `prep/` (RUNT limpio, indicadores, bitácoras y EDA) y `osm/` (accesibilidad, reportes de cruce y límites municipales). **No** incluye los datos originales ni el GeoPackage de OSM, que pesa cientos de MB.

In [19]:
!zip -qr resultados_datawizards.zip prep osm
from google.colab import files
files.download("resultados_datawizards.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>